# Deep RecSys Course
## Домашнее задание 1

### ФИО: <впишите>

### Введение
В этом домашнем задании вы полностью пройдёте базовый пайплайн: подготовка данных → метрики → несколько рекомендательных подходов → итоговый лидерборд.

### Используемые библиотеки
В данном задании потребуются следующие библиотеки:
* [polars](https://pola.rs/) - библиотека для работы с данными (человечество постепенно уходит от `pandas`'а)
* [implicit](https://github.com/benfred/implicit) - библиотека для обучения и применения различных коллаборативных рекомендательных моделей
* [torch](https://pytorch.org/) - no comments
* [gensim](https://radimrehurek.com/gensim/) - обучение **word2vec**

### Данные
Данные лежат в архиве `data.zip`, который состоит из:
* `interactions.parquet` - user-item взаимодействия из датасета Yambda (лайки для 500m версии)
* `embeddings.parquet` - уже пофильтрованные и чуть более плотно запакованные эмбеддинги треков из Yambda
* `artists.parquet` - метаданные айтемов с маппингом в артистов

Скачать архив можно здесь: [ссылка на google disk](https://drive.google.com/file/d/1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS/view?usp=sharing). В следующем блоке мы в любом случае скачиваем датасет, поэтому самостоятельно его можно не качать.

### Guidelines
- Для выполнения ДЗ достаточно использовать Google Collab с T4
- Детерминизм: фиксируйте сиды там, где это важно
- Не используйте данные из теста при обучении/подготовке моделей
- Тест — **последняя неделя** (по timestamp), как описано ниже
- После каждого этапа запускайте проверки внутри ноутбука
- Старайтесь избегать работы с сырыми питоновскими объектами (словарями, списками, интами) там, где можно применить методы из `polars` - они будут в десятки-сотни раз быстрее и читабельней
- Чтобы быстрее обнаруживать, что код написан неоптимально и выполняется слишком долго, старайтесь оборачивать циклы в `tqdm` и выводить progress bar
- Во время отладки кода можно посэмплировать данные для скорости с помощью `data.sample(fraction=0.1, seed=42)`. Для проверки тестов после отладки надо сделать запуски с полным датасетом

### Разбалловка
1. Подготовка данных (1 балл)
2. Оценка качества (1 балл)
3. Топ популярного (1 балл)
4. Рекомендации по артистам (1 балл)
5. Item-to-Item рекомендации (2 балла)
6. Item2Vec (1 балл)
7. Item-based Collaborative Filtering (1 балл)
8. ALS (1 балл)
9. Вопросы (1 балл)

Суммарно - 10 баллов.

In [1]:
!pip install -q gensim
# implicit устанавливается долго, минут 10
!pip install -q implicit

!pip install -q gdown
!gdown --id 1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS -O dataset.zip
!unzip -q dataset.zip
!wget https://raw.githubusercontent.com/iptkachev/DeepRecSys/refs/heads/main/homeworks/hw1/tests.py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 16.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 2.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS
From (redirected): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS&confirm=t&uuid=84088eb2-5f8b-4485-9f1a-caa9b6f19a03
To: /content/dataset.zip
100% 356M/356M [00:02<00:00, 148MB/s]
--2026-03-23 06:57:55--  https://raw.githubusercontent.com/iptkachev/DeepRecSys/refs/heads/main/homeworks/hw1/tests.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185

In [2]:
import sys
sys.path.insert(0, "/content")
import tests

In [3]:
import importlib

# Предполагаем структуру: /content/tests.py или /content/tests/__init__.py

try:
    tests = importlib.import_module("tests")
    print(tests)  # Проверяем импорт
except ImportError as e:
    print(f"Ошибка: {e}")

# Для перезагрузки (если изменили код):
if 'tests' in sys.modules:
    importlib.reload(sys.modules['tests'])

<module 'tests' from '/content/tests.py'>


### 1. Подготовка данных (1 балл)

**Задача:**
1) Считать данные (взаимодействия, эмбеддинги, метаданные).  
2) Оставить только взаимодействия, для которых есть эмбеддинги.  
3) Сделать core фильтрацию: оставить только айтемы с ≥5 взаимодействиями (в ДЗ мы делаем это исключительно для удобства и скорости, в реальной работе так делать не стоит)
4) Поджоинить метаданные (артисты треков) ко всем взаимодействиям. Ремарка: здесь у нас если у одного трека несколько артистов произойдет дублирование прослушивание - в рамках ДЗ мы ничего с этим не делаем, но в реальной жизни такого допускать нельзя.
5) Сделать train-test split: последнюю неделю положить в тест.  
6) Ограничить тест юзерами, у которых есть взаимодействия в трейне.  
7) Подготовить для оценки качества `test_targets: Dict[uid, List[item_id]]`.
8) Оставить только эмбеддинги для core айтемов (остальные пофильтровать).

После этого блока должны существовать `train`, `test`, `embeddings`, `artists`, `test_targets`.

Для этого блока полезны как минимум следующие методы:
* `pl.read_parquet` - для чтения данных
* `.filter, .value_counts` помогут сделать core-фильтрацию
* `df.join(...)` - при джойне метаданных надо использовать `how='left'`, а не `how='inner'`
* `df.join(other, on=some_key, how='semi')` - режим `semi` используется для фильтраций (оставить только те строки из исходного датафрейма, ключ которых присутствует во второй таблице)

In [4]:
from typing import Dict, List, Tuple, Any, Optional
from collections import defaultdict
import os

import numpy as np
import polars as pl

import tests

# Пути к данным (ожидается, что они лежат рядом с ноутбуком)
DATA_DIR = "."
PATH_INTERACTIONS = os.path.join(DATA_DIR, "interactions.parquet")
PATH_EMBEDDINGS = os.path.join(DATA_DIR, "embeddings.parquet")
PATH_ARTISTS = os.path.join(DATA_DIR, "artists.parquet")

# Глобальные параметры
TOPK = 100
CORE_MIN_INTERACTIONS_PER_ITEM = 5
TEST_INTERVAL_SECONDS = 7 * 24 * 60 * 60

# Для воспроизводимости
np.random.seed(42)

data = pl.read_parquet(PATH_INTERACTIONS)
embeddings = pl.read_parquet(PATH_EMBEDDINGS)
artists = pl.read_parquet(PATH_ARTISTS)

###########################
### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ
###########################
interacations_per_item_ids = (
    data
    .group_by("item_id")
    .len()
    .filter(pl.col("len") >= CORE_MIN_INTERACTIONS_PER_ITEM)
)

data = (
    data
    .join(embeddings, on="item_id", how="semi")
    .join(interacations_per_item_ids, on="item_id", how="semi")
    .join(artists, on="item_id", how='left')
)

end_test_interval = data['timestamp'].max()
start_test_interval = end_test_interval - TEST_INTERVAL_SECONDS
train = data.filter(pl.col("timestamp") < start_test_interval)
test = (
    data
    .filter(pl.col("timestamp") >= start_test_interval)
    .join(train, on="uid", how="semi")
)
test_targets = test.group_by("uid").agg(pl.col("item_id")).to_dict(as_series=False)
test_targets = dict(zip(test_targets['uid'], test_targets['item_id']))

embeddings = embeddings.join(interacations_per_item_ids, on="item_id", how="semi")

In [6]:
# Запуск автопроверок (для них необходим файл tests.py)

tests.check_data_split(train=train, test=test, embeddings=embeddings, artists=artists, test_targets=test_targets)

All good! :)


### 2. Оценка качества (1 балл)

#### 2.1 Определения метрик

2.1.1 Пусть для пользователя $u$:

* $G_u \subset \mathcal{I}$ — множество релевантных айтемов (ground truth)
* $R_u = (r_{u,1}, \dots, r_{u,K})$ — упорядоченный список рекомендаций длины $K$

Обозначим индикатор релевантности $I_{u,k} = [ r_{u, k} \in G_u]$. В простонародье его еще часто называют `hits`.

2.1.2 **Hitrate@K** равен единичке, если мы угадали в topK хотя бы один релевантный айтем:
* $
\text{Hitrate@K} = \frac{1}{|U|}
\sum_{u \in U}
\left[ \sum_{k=1}^{K} I_{u,k} > 0 \right]
$

2.1.3 **Recall@K** оценивает долю угаданных релевантных айтемов (от всех релевантных айтемов):
* $
\text{Recall@K} = \frac{1}{|U|}
\sum_{u \in U}
\frac{
\sum_{k=1}^{K} I_{u,k}
}{
\min(|G_u|, K)
}
$

2.1.4 Для подсчета **nDCG@K** нужно сначала посчитать **DCG@K**, затем посчитать **iDCG@K** (DCG в случае идеального ранжирования), затем одно поделить на другое:
* $
\text{DCG@K}(u) = \sum_{k=1}^{K}
\frac{I_{u,k}}{\log_2(k+1)}
$
* $
\text{iDCG@K}(u) = \sum_{k=1}^{\min(|G_u|,K)}
\frac{1}{\log_2(k+1)}
$
* $
\text{nDCG@K} = \frac{1}{|U|}
\sum_{u \in U}
\frac{\text{DCG@K}(u)}{\text{iDCG@K}(u)}
$

2.1.5 **Coverage@K** - это число уникальных айтемов во всех рекомендациях, деленное на размер каталога:

* $
\text{Coverage@K} = \frac{|\bigcup_{u \in U} R_u|}{|\mathcal{I}_{train}|},
$ где $\mathcal{I}_{train}$ — каталог айтемов в train.
* в качестве размера каталога используем количество айтемов, которые нам доступны для рекомендации на момент рекомендации (то есть количество уникальных айтемов в `train`)

#### 2.2 Что нужно сделать
Реализуйте функции:
- `get_metrics(targets, candidates, topk) -> dict(hitrate, recall, ndcg)`
- `evaluate(targets_by_user, candidates_by_user, catalog_size, topk) -> dict(hitrate, recall, ndcg, coverage)`

**Важно:**
* `candidates[uid]` должен иметь длину ровно `topk`
* при подсчете метрики нужно дедуплицировать позитивы; это влияет на значение, на которое мы делим


In [5]:
def get_metrics(targets: List[int], candidates: List[int], topk: int) -> Dict[str, float]:
    ###########################
    ### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ
    ###########################
    metrics = {}
    recall = 0
    hitrate = 0
    dcg = 0
    len_targets = len(set(targets))
    targets_k = set(targets[:topk])
    candidates_k = candidates[:topk]
    for i, c in enumerate(candidates_k):
      indicator = int(c in targets_k)
      hitrate = max(hitrate, indicator)
      recall += indicator
      dcg += indicator / np.log2(i + 2)

    recall = recall / min(topk, len_targets)

    idcg = np.sum([1 / np.log2(i + 2) for i in range(min(topk, len_targets))])
    ndcg = dcg / idcg
    return {"hitrate": hitrate, "recall": recall, "ndcg": ndcg}



def evaluate(
    targets: Dict[int, List[int]],
    candidates: Dict[int, List[int]],
    catalog_size: int,
    topk: int = 100,
) -> Dict[str, float]:
    ###########################
    ### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ
    ###########################
    user_metrics = []
    union_candidates_set = set()
    for uid, t in targets.items():
      c = candidates.get(uid, None)
      if c is not None:
        user_metrics.append(get_metrics(t, c, topk))
        union_candidates_set.update(c)

    mean_metrics = pl.DataFrame(user_metrics).mean().to_dict(as_series=False)
    result = {"coverage": len(union_candidates_set) / catalog_size}
    for name, value in mean_metrics.items():
      result[name] = value[0]
    return result

In [8]:
tests.check_metrics(get_metrics=get_metrics, evaluate=evaluate)

All good! :)


### 3. Топ популярного (1 балл)

Сделайте топ популярных айтемов по train взаимодействиям и посчитайте метрики с помощью `evaluate`.

Полезные методы из `polars`: `.value_counts, .sort, .head, .to_numpy, .tolist, .n_unique`.

**Важно:** в этом пункте не нужно фильтровать для каждого пользователя айтемы, которые уже были в истории пользователя. Нужно просто сделать для всех пользователей один и тот же список кандидатов.

In [6]:
###########################
### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ
###########################
top_pop_k_item_ids = (
    train['item_id']
    .value_counts()
    .sort(by="count", descending=True)
    .head(TOPK)
    ["item_id"]
    .to_list()
)
toppop_test_candidates = (
    test.select("uid")
    .unique()
    .with_columns(item_id=top_pop_k_item_ids)
    .to_dict(as_series=False)
)
toppop_test_candidates = dict(zip(toppop_test_candidates['uid'], toppop_test_candidates['item_id']))
train_catalog_size = train['item_id'].n_unique()
metrics_toppop = evaluate(targets=test_targets, candidates=toppop_test_candidates, catalog_size=train_catalog_size)

In [8]:
metrics_toppop

{'coverage': 0.0006363063687904452,
 'hitrate': 0.11218821770015489,
 'recall': 0.030801353194999367,
 'ndcg': 0.011095930509292347}

In [10]:
# нужно сложить результат метода evaluate в словарь metrics_toppop,
# далее в ноутбуке аналогично ожидаются правильные названия словарей с метриками
tests.check_top_pop(metrics_toppop)

AssertionError: Incorrect recall value

**Важно!** Проверка выше требовала точное совпадение метрик с эталонным ноутбуком, чтобы убедиться в корректности логики обработки данных.
Везде дальше мы требуем метрики выше определенного порога, а не точное совпадение. При этом порог взят с запасом - выбраны ощутимо меньшие значения, чем реальные значения в эталонном ноутбуке. Удачи! :)

### 4. Рекомендации по артистам (1 балл)

Что хотим:
1. Взять последние N лайков пользователя (для моделей и всех подсчетов нужно использовать ТОЛЬКО `train`, тест используется исключительно для оценки качества)
2. Посчитать по ним любимых артистов пользователя (отсортировать по количеству лайков у артиста)
3. Взять у любимых артистов их самые популярные треки (чтобы посчитать популярность треков, нужно использовать train)
4. Оставить те, которые пользователь еще не видел (не лайкал)
5. Их порекомендовать

Фактически, мы формируем рекомендации по счётчикам - учитываем "сколько раз пользователь лайкал артиста" и "сколько раз трек артиста был лайкнут".

В этом методе 100 кандидатов найдётся далеко не для всех пользователей, поэтому нам нужно дополнить рекомендации до 100 каким-то дополнительным методом; то есть сделать fallback. В данном случае предлагается делать fallback на `top_pop` рекомендации:
* если у пользователя набралось меньше 100 рекомендаций, то идём по top_pop и добавляем кандидатов в рекомендации, пока не заполним до 100
* при этом добавляем только треки, которых не было в уже отобранных до fallback'а рекомендациях

**Важно:** есть два варианта, как отрезать `PER_ARTIST_LIMIT` для каждого артиста - с учётом уже прослушанного или без. Вариант, который даёт нужные высокие значения метрик - с отрезанием без учёта прослушанного. То есть если в top 20 артиста пользователь всё уже лайкнул, мы его пропускаем.

In [7]:
# именно такие значения ожидает автопроверка tests.check_artist_recs
N_LAST_EVENTS = 100  # n последних взаимодействий
PER_ARTIST_LIMIT = 30  # берем для рекомендаций 20 айтемов из каждого любимого артиста

# эту функцию будем переиспользовать в следующих пунктах
def fallback_to_toppop(cands_by_uid: Dict[int, List[int]], top_pop: np.ndarray, topk: int) -> Dict[int, List[int]]:
    ###########################
    ### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ
    ###########################
    for _, cands in cands_by_uid.items():
        if len(cands) < topk:
            for tp in top_pop:
                if tp not in cands:
                    cands.append(tp)
                if len(cands) == topk:
                    break
    return cands_by_uid

###########################
### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ
###########################

last_n_events_per_user = (
    train
    .sort(["uid", "timestamp", "item_id"], descending=[False, True, False])
    .group_by("uid")
    .head(N_LAST_EVENTS)
)

fav_artists_per_user = (
    last_n_events_per_user
    .group_by(["uid", "artist_id"])
    .len()
    .rename({"len": "len_artist_id"})
)

pop_artists = (
    train
    .group_by(["artist_id", "item_id"])
    .len()
    .sort(["artist_id", "len"], descending=[False, True])
    .group_by("artist_id")
    .head(PER_ARTIST_LIMIT)
    .select("artist_id", "item_id", pl.col("len").alias("len_item_id"))
)

recart_test_candidates = (
    fav_artists_per_user
    .join(test, on="uid", how="semi")
    .join(pop_artists, on="artist_id")
    .join(last_n_events_per_user, on=["uid", "item_id"], how="anti")
    .sort(["uid", "len_artist_id", "len_item_id"], descending=[False, True, True])  # великий рандом улучшает метрики
    .group_by("uid")
    .head(TOPK)
    .group_by("uid")
    .agg(pl.col("item_id"))
    .to_dict(as_series=False)
)
recart_test_candidates = dict(zip(recart_test_candidates['uid'], recart_test_candidates['item_id']))
recart_test_candidates = fallback_to_toppop(recart_test_candidates, top_pop_k_item_ids, TOPK)
metrics_artist = evaluate(targets=test_targets, candidates=recart_test_candidates, catalog_size=train_catalog_size)

In [10]:
metrics_artist

{'coverage': 0.6255591542215746,
 'hitrate': 0.20377792027359196,
 'recall': 0.056747277721776845,
 'ndcg': 0.021692358071724076}

In [23]:
tests.check_artist_recs(metrics_artist)

All good! :)


### 5. Item-to-Item рекомендации (2 балла)

Здесь предлагается реализовать классический item-to-item алгоритм:
1. Считаем для каждого айтема список похожих айтемов
2. Берём последние N взаимодействий пользователя, для каждого вытаскиваем список похожих
3. Агрегируем кандидатов из списка похожих, учитывая суммарные похожести (если айтем встретился в нескольких списках кандидатов у пользователя, суммируем похожесть из каждого)
4. Отфильтровываем все, что уже было у пользователя в истории
5. Оставляем только topk кандидатов

Реализация должна состоять из двух функций:
1) `get_similar_items_gpu(item_ids, item_embeddings, topk_sim)` - для каждого айтема находим top similar items по **косинусу**
* используем pytorch (и GPU), иначе подсчет будет очень долгим
* создаем тензор с эмбеддингами, переводим на GPU, $l_2$-нормализуем эмбеддинги (`torch.nn.functional.normalize`)
* идём батчами по эмбеддингам, для каждого батча с помощью `torch.topk` считаем честный топ похожих; бачти - это важно! Батчевание сильно влияет на скорость
* запоминаем эту информацию в словаре вида `item_id -> [(cand_item_id, similarity), ...]`

2) `get_candidates_item2item(interactions, similar_items, ...)` - для каждого пользователя агрегируем похожести по последним N взаимодействиям.
* нужно буквально реализовать вышеописанный алгоритм "берём последние N взаимодействий, суммируем похожести всех полученных похожих айтемов"
* для написания этой функции лучше не пытаться сделать супер оптимальный код через `polars` (у него очень большое пиковое потребление оперативной памяти); в данном случае проще манипулировать словарями, списками, etc
* предлагается использовать `heapq.nlargest` для ускорения поиска `topk` элементов среди уже подсчитанных суммарных похожестей (раза в полтора-два быстрее, чем делать полную сортировку всех кандидатов со всех списков)

В пункте 2 нужно дополнительно поддержать **time decay** (затухание по времени):
* свежие взаимодействия должны вносить более высокий вклад, поэтому при суммировании похожестей для конкретного кандидата мы учитываем свежесть события, из списка которого берём похожесть
* вес события $2^{-\frac{L-t}{\tau}}$, где $L$ - длина истории пользователя, $t$ - позиция события в истории (начиная с единички), $\tau$ - период полураспада, то есть насколько быстро затухает сигнал от событий. Например, при $\tau = 10$ вес события падает в два раза, если оно идет на 10 позиций раньше в истории
* Получаем $\text{score}(u, j) =
\sum_{t=1}^{L}
2^{- \frac{L - t}{\tau}}
\cdot
s(i_{u,t}, j)
$

**Важно:** при использовании метода `get_similar_items_gpu` набираем не 100 кандидатов, а в два раза больше (2 * topk = 200), чтобы после фильтрации по истории пользователя и слияния всех списков у нас было больше шансов набрать `topk`.

Полезные методы из `polars`: `.to_torch()`, `.to_numpy()` - позволяют сразу получить тензоры для айдишников / эмбеддингов

In [12]:
import torch
import torch.nn.functional as F
import heapq
from tqdm import tqdm
from collections import defaultdict


def add_to_heap(heap, item, max_size):
    if len(heap) < max_size:
        heapq.heappush(heap, item)
    else:
        heapq.heappushpop(heap, item)

def get_similar_items_gpu(
    item_ids: np.ndarray,
    item_embeddings: np.ndarray,
    block: int = 1024,
    topk: int = 200,
    device: str = "cuda",
) -> Dict[int, List[Tuple[int, float]]]:
    """
    Возвращает словарь item_id -> [(cand_item_id, similarity), ...] длины topk
    """
    ###########################
    ### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ
    ###########################
    similar_items = {}
    item_embeddings = F.normalize(torch.FloatTensor(item_embeddings).to(device))
    for i in tqdm(range(0, len(item_ids), block), total=len(item_ids) // block):
        item_heap = []
        chunk = item_embeddings[i: i + block]
        topk_chunk_sim, topk_chunk_ind = torch.topk(chunk @ item_embeddings.T, topk + 1)
        topk_chunk_sim, topk_chunk_ind = topk_chunk_sim[:, 1:], topk_chunk_ind [:, 1:] # exclude sim with self
        for i, item in enumerate(item_ids[i: i + block]):
            item_sim = topk_chunk_sim[i].numpy().tolist()
            item_sim_items = item_ids[topk_chunk_ind[i]].tolist()
            similar_items[int(item)] = list(zip(item_sim_items, item_sim))
    return similar_items


def get_candidates_item2item(
    interactions: pl.DataFrame,
    similar_items: Dict[int, List[Tuple[int, float]]],
    n_last: int = 30,
    half_life_frac: float = 0.5,
    topk: int = 100,
) -> Dict[int, List[int]]:
    """
    half_life_frac интерпретируется как доля n_last: half_life = half_life_frac * n_last.
    То есть в реальной формуле вам нужно домножить параметр half_life_frac на n_last
    w(r) = 2^{-r / half_life}, где r=0 для самого последнего события.
    """
    ###########################
    ### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ
    ###########################
    last_items = (
      interactions
      .select("uid", "item_id", "timestamp")
      .sort(["uid", "timestamp"])
      .group_by("uid")
      .tail(n_last)
      .with_columns(pl.int_range(1, pl.len() + 1).rank().over("uid").alias("rank"))
      .with_columns(pl.lit(2.0).pow(pl.col("rank") / (half_life_frac * n_last)).alias("weight"))
      .group_by("uid")
      .agg(pl.struct("item_id", "weight").implode().alias("item_id_pairs"))
      .to_dict(as_series=False)
  )

    map_uid_to_heap_item_sums = {}
    for uid, item_id_pairs in tqdm(zip(last_items["uid"], last_items["item_id_pairs"]), total=len(last_items)):
        uid_cand_sum = defaultdict(float)

        for pair in item_id_pairs:
            # для каждого айтема находим топ похожих на этот айтем
            for cand_item_id, cand_sim in similar_items.get(pair['item_id'], []):
                uid_cand_sum[cand_item_id] += cand_sim * pair['weight']
        all_item_id_history = set(pair['item_id'] for pair in item_id_pairs)
        # исключаем из рекомендаций то, что уже было в истории n_last
        uid_cand_sum = {
            cand_id: cand_sum for cand_id, cand_sum in uid_cand_sum.items()
            if cand_id not in all_item_id_history
        }
        # берем topk рекомендаций
        uid_cand_sum = heapq.nlargest(topk, uid_cand_sum.items(), key=lambda x: x[1])
        if uid_cand_sum:
            map_uid_to_heap_item_sums[uid] = [cand_id for cand_id, _ in uid_cand_sum]
    return map_uid_to_heap_item_sums

In [ ]:
similar_items = get_similar_items_gpu(
    embeddings['item_id'].to_numpy(),
    embeddings['embed'].to_numpy(),
    block= 1024,
    topk= TOPK * 2,
    device= "cpu",
)


i2i_test_candidates = get_candidates_item2item(
    train,
    similar_items,
    n_last = 30,
    half_life_frac = 0.5,
    topk= TOPK,
)

154it [04:10,  1.62s/it]


In [ ]:
###########################
### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ
###########################
i2i_test_candidates = fallback_to_toppop(i2i_test_candidates, top_pop_k_item_ids, TOPK)
metrics_i2i = evaluate(targets=test_targets, candidates=i2i_test_candidates, catalog_size=train_catalog_size)

In [ ]:
metrics_i2i

{'coverage': 0.9358794072169868,
 'hitrate': 0.1257277145756556,
 'recall': 0.031211359053422656,
 'ndcg': 0.010975645379105397}

In [ ]:
tests.check_i2i_recs(metrics_i2i)

All good! :)


### 6. Item2Vec (1 балл)

Item-to-item можно использовать с буквально любыми похожестями айтемов.

Теперь давайте попробуем сами обучить эмбеддинги айтемов, посчитать похожести и опять применить item-to-item.

Будем использовать подход Item2Vec - применим **word2vec** к последовательностям айтемов вместо последовательностей слов.

Для этого предлагается использовать `gensim`:
* нужно создать `corpus` из списка списков строк вида `[['1', '2'], ['3', '4', '5']]`, в котором строки - это буквально строки айдишников айтемов, а внутренние списки - это сгруппированные взаимодействия пользователя (хронологически отсортированные)
* Вызвать метод `gensim.models.Word2Vec`. Предлагаемые настройки `vector_size=100, window=5, min_count=1, sg=0, epochs=5`

После обучения, эмбеддинги можно достать с помощью `w2v.wv.vectors`, а соответствующие айдишники - с помощью `w2v.wv.index_to_key`.

Далее предлагается применить пайплайн из прошлого пункта: `get_similar_items_gpu` + `get_candidates_item2item`

In [ ]:
from gensim.models import Word2Vec

###########################
### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ
###########################

sentences = (
    train
    .sort(["uid", "timestamp"])
    .group_by("uid")
    .agg(pl.col("item_id"))
    ['item_id']
    .to_list()
)
w2v_model = Word2Vec(sentences=sentences, vector_size=100, window=5, min_count=1, sg=0, epochs=5)

In [ ]:
w2v_similar_items = get_similar_items_gpu(
    np.array(w2v_model.wv.index_to_key),
    w2v_model.wv.vectors,
    block= 1024,
    topk= TOPK * 2,
    device= "cpu",
)

w2v_test_candidates = get_candidates_item2item(
    train,
    w2v_similar_items,
    n_last = 30,
    half_life_frac = 0.5,
    topk= TOPK,
)

w2v_test_candidates = fallback_to_toppop(w2v_test_candidates, top_pop_k_item_ids, TOPK)
metrics_w2v = evaluate(targets=test_targets, candidates=w2v_test_candidates, catalog_size=train_catalog_size)

154it [03:31,  1.37s/it]


In [ ]:
metrics_w2v

{'coverage': 0.7895671207773118,
 'hitrate': 0.18346418843134113,
 'recall': 0.04613451546166496,
 'ndcg': 0.01711937779032364}

In [ ]:
tests.check_w2v_recs(metrics_w2v)

### 7. Item-based Collaborative Filtering (1 балл)

Теперь попробуем применить тот же item-to-item подход, но используя в качестве векторов большие разреженные векторы из user-item матрицы. Для подсчета разреженных близостей наш GPU пайплайн не подходит (не умеет работать с разреженными данными), поэтому будем использовать библиотеку `implicit`

1. Сначала нужно собрать CSR user-item матрицу из наших `train` взаимодействий с помощью `scipy.sparse.csr_matrix`
* предлагается использовать формат вида `csr_matrix((ones, (user_ids, item_ids)), shape=(num_users, num_items))`
* чтобы просуммировать взаимодействия с одними и теми же айтемами можно использовать `.sum_duplicates`
* также понадобится от исходных айдишников (которые необязательно от 1 до `num_items / num_users`) перейти к компактным айдишникам, сделав маппинг `{old_item_id: new_item_id}`. Простой вариант -- сделать это с помощью словаря, более быстрый - использовать метод вида `train.select("uid").unique().with_row_index()` (аналогично для айтемов). И еще очень хак для ускорения - чтобы в таблице со старым `old_item_id` получить `new_item_id`, достаточно сделать джойн: `interactions.join(mapping_table, on='old_item_id', how='left')`, где название `old_item_id` зависит от имплементации (скорее всего это будет просто `item_id`)

2. Чтобы с помощью `implicit` получить списки похожих, нужно использовать комбинацию из `CosineRecommender.fit`, и `.similar_items`

3. Чтобы сформировать рекомендации, используем нашу функцию `get_candidates_item2item`

**Warning:** неаккуратно написанный код в этом пункте может переполнить оперативную память.

In [8]:
import numpy as np
from scipy.sparse import csr_matrix

from implicit.nearest_neighbours import CosineRecommender, tfidf_weight

uid_mapping = train.select("uid").unique().with_row_index().select(pl.col("index").alias("index_uid"), "uid")
item_id_mapping = train.select("item_id").unique().with_row_index().select(pl.col("index").alias("index_item_id"), "item_id")
train_w_mapping = train.join(uid_mapping, on="uid", how="inner").join(item_id_mapping, on="item_id", how="inner")

ones = np.ones(train_w_mapping.shape[0])
user_ids = train_w_mapping['index_uid']
NUM_USERS = user_ids.unique().shape[0]
item_ids = train_w_mapping['index_item_id']
NUM_ITEMS = item_ids.unique().shape[0]
sparse_matrix = csr_matrix((ones, (user_ids, item_ids)), shape=(NUM_USERS, NUM_ITEMS))

In [9]:
def create_cf_similar_items(cosine_recom, df_item_mapping, topk):
    dict_item_mapping = item_id_mapping.to_dict(as_series=False)
    dict_item_mapping = dict(zip(dict_item_mapping["index_item_id"], dict_item_mapping["item_id"]))
    cf_similar_items = {}
    for index_item_id, item_id in tqdm(dict_item_mapping.items(), total=len(dict_item_mapping)):
        similar_index_item_ids, similar_value_item_ids = cosine_recom.similar_items(index_item_id, topk)
        cf_similar_items[item_id] = [(dict_item_mapping[index], value)
                                     for index, value in zip(similar_index_item_ids, similar_value_item_ids)]
    return cf_similar_items


def run_cosine(X: csr_matrix, half_life_frac, n_last):
    cosine_recom = CosineRecommender(2 * TOPK)
    cosine_recom.fit(X)
    cf_similar_items = create_cf_similar_items(cosine_recom, item_id_mapping, 2 * TOPK)

    cf_test_candidates = get_candidates_item2item(
        train,
        cf_similar_items,
        n_last = n_last,
        half_life_frac = half_life_frac,
        topk= TOPK,
    )
    cf_test_candidates = fallback_to_toppop(cf_test_candidates, top_pop_k_item_ids, TOPK)
    test_uid = set(test_targets.keys())
    cf_test_candidates = {uid: item_ids for uid, item_ids in cf_test_candidates.items() if uid in test_uid}
    metrics_cf = evaluate(targets=test_targets, candidates=cf_test_candidates, catalog_size=train_catalog_size)
    return metrics_cf

In [77]:
metrics_cf = run_cosine(sparse_matrix, 0.4, 30)

/usr/local/lib/python3.12/dist-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.07705426216125488 seconds
  warnings.warn(


  0%|          | 0/157157 [00:00<?, ?it/s]

100%|██████████| 157157/157157 [01:13<00:00, 2128.27it/s]
81020it [05:36, 241.03it/s]


In [78]:
metrics_cf

{'coverage': 0.7854374924438619,
 'hitrate': 0.17112642204774875,
 'recall': 0.052956556738181705,
 'ndcg': 0.02133846954300215}

In [36]:
tests.check_cf_recs(metrics_cf)

All good! :)


##### 7.2 TF-IDF

А теперь щепотка магии - с помощью `implicit.nearest_neighbours.tfidf_weight` модифицируем user-item матрицу и получим более высокие метрики.

Все, что нужно - это применить этот метод к матрице и затем проделать все те же операции, что и раньше (с вычислением похожестей и т.д.)

In [19]:
X_tfidf = tfidf_weight(sparse_matrix)

In [15]:
metrics_cf_tfidf = run_cosine(X_tfidf, 0.1, 100)

/usr/local/lib/python3.12/dist-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.07599139213562012 seconds
  warnings.warn(


  0%|          | 0/157157 [00:00<?, ?it/s]

100%|██████████| 157157/157157 [01:06<00:00, 2351.30it/s]
81020it [11:36, 116.40it/s]


In [18]:
metrics_cf_tfidf

{'coverage': 0.8309652131308183,
 'hitrate': 0.25634246648507186,
 'recall': 0.08148319484044698,
 'ndcg': 0.03187897113316971}

In [17]:
tests.check_tfidf_recs(metrics_cf_tfidf)

AssertionError: Too low hitrate value

### 8. ALS (1 балл)

С помощью `implicit.als.AlternatingLeastSquares` над той же самой разреженной user-item матрицей можно обучить ALS, что вам и предлагается сделать.

Чтобы сформировать рекомендации, наша функция `get_candidates_item2item` не нужна - достаточно использовать метод `model.recommend`.

Нужно обучить ALS с двумя версиями user-item матриц - исходной и tfidf-модифицированной.

In [15]:
from implicit.als import AlternatingLeastSquares


def predict_als(fitted_als, test_uid, als_sparse_matrix_to_rec):
    # чтобы сформировать топ последних айтемов для предсказания для каждого юзера
    # train_last_n_per_user = (
    #     train
    #     .select("uid", "item_id", "timestamp")
    #     .sort(["uid", "timestamp"])
    #     .group_by("uid")
    #     .tail(n_last_items)
    #     .join(uid_mapping, on="uid", how="inner")
    #     .join(item_id_mapping, on="item_id", how="inner")
    # )

    # ones = np.ones(train_last_n_per_user.shape[0])
    # user_ids = train_last_n_per_user['index_uid']
    # item_ids = train_last_n_per_user['index_item_id']
    # als_sparse_matrix_to_rec = csr_matrix((ones, (user_ids, item_ids)),
    #                                       shape=(NUM_USERS, NUM_ITEMS))

    map_uid_to_index_uid = dict(zip(
        uid_mapping['uid'].to_list(),
        uid_mapping['index_uid'].to_list()
    ))
    map_index_item_id_to_item_id = dict(zip(
        item_id_mapping['index_item_id'].to_list(),
        item_id_mapping['item_id'].to_list()
    ))

    map_uid_to_item_ids = {}
    for uid in tqdm(test_uid, total=len(test_uid)):
        index_uid = map_uid_to_index_uid[uid]
        reqs, _ = fitted_als.recommend(index_uid, als_sparse_matrix_to_rec[index_uid], N=TOPK)
        map_uid_to_item_ids[uid] = [
            map_index_item_id_to_item_id[index_item]
            for index_item in reqs
        ]
    return map_uid_to_item_ids


def run_als(X: csr_matrix, factors: int = 100, reg: float = 0.5, alpha: float = 0.1, iters: int = 20, n_last_items=100):
    ###########################
    ### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ
    ###########################
    als = AlternatingLeastSquares(factors, reg, alpha, iterations=iters)
    als.fit(X)

    test_uid = set(test_targets.keys())
    als_test_candidates = predict_als(als, test_uid, X)
    als_test_candidates = fallback_to_toppop(als_test_candidates, top_pop_k_item_ids, TOPK)
    als_test_candidates = {uid: item_ids for uid, item_ids in als_test_candidates.items() if uid in test_uid}
    metrics_als = evaluate(targets=test_targets, candidates=als_test_candidates, catalog_size=train_catalog_size)
    return metrics_als

In [24]:
metrics_als_raw = run_als(sparse_matrix, factors=64, alpha=80, reg=0.1, iters=20)
metrics_als_tfidf = run_als(X_tfidf.tocsr(), factors=100, alpha=5, reg=5, iters=20)

  0%|          | 0/20 [00:00<?, ?it/s]

100%|██████████| 37446/37446 [03:58<00:00, 157.14it/s]


In [32]:
metrics_als_raw, metrics_als_tfidf

({'coverage': 0.21325171643642982,
  'hitrate': 0.2858783314639748,
  'recall': 0.08899741451826175,
  'ndcg': 0.032578238806754},
 {'coverage': 0.2292484585478216,
  'hitrate': 0.3036372376221759,
  'recall': 0.09724201016808107,
  'ndcg': 0.035620388266363386})

In [33]:
tests.check_als_recs(metrics_als_raw, metrics_als_tfidf)

All good! :)


### 9. Лидерборд и выводы

Собираем таблицу со всеми методами и метриками.

Добавьте 5–10 строк выводов к экспериментам: что работает лучше и почему.


In [41]:
leaderboard = pl.DataFrame([
    {"method": "TopPop", **metrics_toppop},
    {"method": "User-Artist", **metrics_artist},
    {"method": "Item2Item (dataset emb)", **metrics_i2i},
    {"method": "Item2Vec (w2v)", **metrics_w2v},
    {"method": "CF Cosine (raw)", **metrics_cf},
    {"method": "CF Cosine (tf-idf)", **metrics_cf_tfidf},
    {"method": "ALS (raw)", **metrics_als_raw},
    {"method": "ALS (tf-idf)", **metrics_als_tfidf},
])

leaderboard = leaderboard.sort(["recall", "ndcg"], descending=True)
leaderboard

method,coverage,hitrate,recall,ndcg
str,f64,f64,f64,f64
"""ALS (tf-idf)""",0.229248,0.303637,0.097242,0.03562
"""ALS (raw)""",0.213252,0.285878,0.088997,0.032578
"""CF Cosine (tf-idf)""",0.830965,0.256342,0.081483,0.031879
"""User-Artist""",0.625559,0.203778,0.056747,0.021692
"""CF Cosine (raw)""",0.785437,0.171126,0.052957,0.021338
"""Item2Vec (w2v)""",0.789567,0.183464,0.046135,0.017119
"""Item2Item (dataset emb)""",0.935879,0.125728,0.031211,0.010976
"""TopPop""",0.000636,0.112188,0.030801,0.011096


### Вопросы на понимание (1 балл)

1) Почему для кандидатов/метрик важно исключать айтемы, которые пользователь уже видел?  
2) Почему при джойне метаданных (да и вообще почти при любом джойне) нужно использовать how='left', а не how='inner'?
3) В рекомендациях по артистам (с помощью счётчиков) не для всех пользователей может найтись нужное количество кандидатов. Почему?
4) Почему tf-idf улучшает item-based CF? На саму функцию можно посмотреть через `tfidf_weight??`
5) В чем принципиальное отличие между item-to-item методом и методом, при котором мы получаем эмбеддинг пользователя, сложив эмбеддинги его последних взаимодействий, и затем ищем ближайшие эмбеддинги айтемов?
5) Почему ALS выиграл у чистого cosine-item2item?

Ответы - текстом